In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import HTML, Markdown, display

from ne_syntax_consistency.spans import LABELS, span_trees, support_table
from ne_syntax_consistency.syntax_vis import Style
from ne_syntax_consistency.viz import record_svg

STYLE = Style()


def _project_root() -> Path:
    # Jupyter käivitab kerneli notebooki kaustas; otsime üles pyproject.toml-i
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("pyproject.toml not found")


DATA = _project_root() / "data"

In [ ]:
SPAN_FILL, SPAN_ARC, CONTEXT_ARC, NE_FILL = (
    STYLE.highlight_fill,
    STYLE.highlight_arc,
    STYLE.context_arc,
    STYLE.box_fill,
)
display(
    Markdown(f"""
# NE spannide sisemised puud

Iga NE span annab täpselt ühe mustri: tema sõnade ja nendevaheliste kaarte puu
(`scripts/span_trees.py`, `ne_syntax_consistency.spans`). Mustri tugi = spannide arv,
mille terve sisemine puu on just see muster; alammustreid ei loeta. Uurime mustreid,
mille tugi on vähemalt piirväärtus (vaikimisi 10), ja iga mustri kohta juhuslikke näitelauseid.

Joonisel: <span style="background:{SPAN_FILL};padding:0 4px">span</span>,
<b style="color:{SPAN_ARC}">paks kaar</b> = spanni sisene kaar (muster),
<b style="color:{CONTEXT_ARC}">katkendkaar</b> = spanni juure seos välise ülemaga.
Teised NE-d on <span style="background:{NE_FILL};padding:0 4px">helesinisel taustal</span>, tüüp kasti all.
Sõna all on sõnaliik ja kääne (`morph_analysis`); "…" = peidetud sõnad.
""")
)

In [ ]:
# Parameetrid. Valim ekspordi vajadusel:
#   uv run python scripts/export_sample.py --sentences 200000 --seed 1
SAMPLE = "sample_200000_s1.parquet"
LABEL = "type"  # sõnade märgend mustris, üks LABELS-ist
MIN_SUPPORT = 10  # min tugi
TOP_N = 10  # mustreid tüübi kohta
N_EXAMPLES = 3  # näiteid mustri kohta
WINDOW = 5  # sõnu iga spanni sõna ümber (0 = ainult span ja ülemad, None = terve lause)
SEED = 1

assert LABEL in LABELS, list(LABELS)
print("Saadaval:", sorted(p.name for p in DATA.glob("sample_*.parquet")))

In [ ]:
sentences = {
    s["text_id"]: s
    for s in pq.read_table(
        DATA / SAMPLE,
        columns=[
            "text_id",
            "words",
            "pos",
            "form",
            "head",
            "deprel",
            "ne_tag",
            "ne_words",
        ],
    ).to_pylist()
}
len(sentences)

In [ ]:
spans = pd.DataFrame(row for s in sentences.values() for row in span_trees(s, LABEL))
table = support_table(spans)

In [ ]:
# Iga tüübi top-N mustrit, mille tugi >= min tugi
top = (
    table[table["support"] >= MIN_SUPPORT]
    .groupby("nertag", group_keys=False)
    .head(TOP_N)
)
_summary = (
    table.assign(frequent=table["support"] >= MIN_SUPPORT)
    .groupby(["nertag", "frequent"])
    .agg(patterns=("pattern", "size"), spans=("support", "sum"))
    .unstack("frequent", fill_value=0)
)
display(Markdown(f"### Mustrite arv tüübi kaupa (tugi ≥ {MIN_SUPPORT}: True)"))
display(_summary)
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(top[["nertag", "support", "share_of_type", "n_roots", "pattern"]])

In [ ]:
def show_examples(nertag):
    """Joonistab tüübi top-mustrid koos juhuslike näitelausetega."""
    display(Markdown(f"## {nertag}"))
    for p in top[top["nertag"] == nertag].itertuples():
        display(
            Markdown(
                f"#### `{p.pattern}`\n\ntugi **{p.support}** "
                f"({p.share_of_type:.1%} tüübi spannidest), juuri {p.n_roots}"
            )
        )
        matching = spans[
            (spans["nertag"] == p.nertag) & (spans["pattern"] == p.pattern)
        ]
        examples = matching.sample(n=min(N_EXAMPLES, len(matching)), random_state=SEED)
        for row in examples.itertuples():
            s = sentences[row.text_id]
            svg = record_svg(s, s["ne_words"][row.ne_index], WINDOW, STYLE)
            display(Markdown(f"`{row.text_id}` · {' '.join(s['words'])}"))
            display(HTML(f'<div style="overflow-x:auto">{svg}</div>'))


sorted(top["nertag"].unique())

In [ ]:
show_examples("ORG")

In [ ]:
show_examples("PER")

In [ ]:
show_examples("LOC")